In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor  # NEW: Added XGBoost import
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# 1. Load Datasets
train_path = "/kaggle/input/competitions/home-data-for-ml-course/train.csv"
test_path = "/kaggle/input/competitions/home-data-for-ml-course/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Original Train shape: {train_df.shape}")
print(f"Original Test shape: {test_df.shape}")

# NEW: Outlier Removal (GrLivArea > 4000 sq ft sold cheap)
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index)

test_ids = test_df['Id']

quality_keys = {'Ex', 'Gd', 'TA', 'Fa', 'Po'}
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}

# 2. Reusable Cleaning Function
def clean_data(df):
    df = df.copy()
    
    # Ordinal Quality Mapping
    ordinal_cols = [
        col for col in df.select_dtypes(include=['object']).columns
        if quality_keys.intersection(set(df[col].dropna().unique()))
    ]
    for col in ordinal_cols:
        df[col] = df[col].map(quality_map).fillna(0).astype(int)
        
    # Numerical Median Imputations
    df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    
    # Mode Imputation
    most_frequent_electrical = df['Electrical'].mode()[0] if not df['Electrical'].mode().empty else 'SBrkr'
    df['Electrical'] = df['Electrical'].fillna(most_frequent_electrical)
    
    # Text Fill with 'None'
    text_cols_to_fill = [
        'Alley', 'MasVnrType', 'BsmtFinType1', 'BsmtFinType2',
        'GarageType', 'GarageFinish', 'Fence', 'MiscFeature'
    ]
    existing_text_cols = [c for c in text_cols_to_fill if c in df.columns]
    df[existing_text_cols] = df[existing_text_cols].fillna('None')
    
    # Numeric Year Fill
    if 'GarageYrBlt' in df.columns:
        df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)
        
    # Catch any remaining missing values
    num_cols = df.select_dtypes(include=['number']).columns
    obj_cols = df.select_dtypes(include=['object']).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[obj_cols] = df[obj_cols].fillna('None')
    
    # Feature Engineering
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch'] + df['WoodDeckSF']
    
    return df

df_train_clean = clean_data(train_df)
df_test_clean = clean_data(test_df)

# One-Hot encode the data
df_train_encoded = pd.get_dummies(df_train_clean)
df_test_encoded = pd.get_dummies(df_test_clean)

y = df_train_encoded['SalePrice']
X = df_train_encoded.drop(columns=['SalePrice', 'Id'])
X_test = df_test_encoded.drop(columns=['Id'])

# Align to ensure matching features
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# Target Log Transformation
y_log = np.log1p(y)

# Validation split using y_log
X_train_split, X_val_split, y_train_split_log, y_val_split_log = train_test_split(
    X, y_log, test_size=0.2, random_state=0
)
y_val_actual = np.expm1(y_val_split_log)  # Reverted back to actual dollars for MAE evaluation

# Side-by-Side Evaluation (Random Forest vs XGBoost vs Blend)
rf = RandomForestRegressor(n_estimators=300, random_state=1)
rf.fit(X_train_split, y_train_split_log)
rf_val_preds = np.expm1(rf.predict(X_val_split))
rf_mae = mean_absolute_error(y_val_actual, rf_val_preds)

xgb = XGBRegressor(
    n_estimators=1000, 
    learning_rate=0.02, 
    max_depth=4, 
    subsample=0.7, 
    colsample_bytree=0.7, 
    random_state=42
)
xgb.fit(X_train_split, y_train_split_log)
xgb_val_preds = np.expm1(xgb.predict(X_val_split))
xgb_mae = mean_absolute_error(y_val_actual, xgb_val_preds)

blend_val_preds = (rf_val_preds * 0.5) + (xgb_val_preds * 0.5)
blend_mae = mean_absolute_error(y_val_actual, blend_val_preds)

print("\n--- Validation Results (MAE) ---")
print(f"Random Forest MAE: ${rf_mae:,.2f}")
print(f"XGBoost MAE:       ${xgb_mae:,.2f}")
print(f"50/50 Blend MAE:   ${blend_mae:,.2f}")

# NEW: Auto-select winning strategy for full-data fit & submission
if blend_mae < min(rf_mae, xgb_mae):
    print("\n-> Winner: Ensembled Blend")
    rf_final = RandomForestRegressor(n_estimators=300, random_state=1).fit(X, y_log)
    xgb_final = XGBRegressor(n_estimators=1000, learning_rate=0.02, max_depth=4, subsample=0.7, colsample_bytree=0.7, random_state=42).fit(X, y_log)
    
    rf_test_preds = np.expm1(rf_final.predict(X_test))
    xgb_test_preds = np.expm1(xgb_final.predict(X_test))
    final_test_preds = (rf_test_preds * 0.5) + (xgb_test_preds * 0.5)
elif xgb_mae < rf_mae:
    print("\n-> Winner: XGBoost")
    xgb_final = XGBRegressor(n_estimators=1000, learning_rate=0.02, max_depth=4, subsample=0.7, colsample_bytree=0.7, random_state=42).fit(X, y_log)
    final_test_preds = np.expm1(xgb_final.predict(X_test))
else:
    print("\n-> Winner: Random Forest")
    rf_final = RandomForestRegressor(n_estimators=300, random_state=1).fit(X, y_log)
    final_test_preds = np.expm1(rf_final.predict(X_test))

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_test_preds})
submission.to_csv('submission.csv', index=False)
print("Submission successfully saved to 'submission.csv'!")

Original Train shape: (1460, 81)
Original Test shape: (1459, 80)

--- Validation Results (MAE) ---
Random Forest MAE: $16,374.02
XGBoost MAE:       $13,960.60
50/50 Blend MAE:   $14,842.64

-> Winner: XGBoost
Submission successfully saved to 'submission.csv'!
